<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/learn-ml-algorithms-2017-notebooks/blob/main/12-dimensionality-reduction.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/learn-ml-algorithms-2017/) collection._

# Chapter 12 — Dimensionality Reduction

Companion to [the chapter](https://www.ai.biz/books/learn-ml-algorithms-2017/dimensionality-reduction/).

Why many features make distance stop meaning anything, PCA built from the covariance matrix up, and t-SNE with its traps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
X, y = digits.data, digits.target
print(f'{X.shape[0]} handwritten digits, each {X.shape[1]} pixels (8 x 8), values 0 to 16')

## 1. The curse: in many dimensions, everything is far from everything

In [ ]:
rng = np.random.default_rng(0)
print(f'{"dims":>5} {"nearest":>8} {"farthest":>9} {"ratio":>6}')
for d in [1, 2, 10, 100, 1000]:
    P = rng.uniform(0, 1, (500, d))
    dist = np.linalg.norm(P - P[0], axis=1)[1:]              # from the first point to every other
    print(f'{d:>5} {dist.min():>8.3f} {dist.max():>9.3f} {dist.max() / dist.min():>6.2f}')

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

# k-NN on the digits, with more and more columns of noise added
for extra in [0, 64, 256, 1024]:
    Xn = np.column_stack([X, rng.normal(0, 4, (len(X), extra))]) if extra else X
    acc = cross_val_score(KNeighborsClassifier(5), Xn, y, cv=5).mean()
    print(f'64 real pixels + {extra:>4} noise columns: k-NN accuracy {acc:.3f}')

## 2. PCA from the eigenvector up

In [ ]:
Xc = X - X.mean(axis=0)                                      # centre every pixel
cov = Xc.T @ Xc / (len(Xc) - 1)                              # 64 x 64 covariance matrix
eigenvalues, eigenvectors = np.linalg.eigh(cov)             # eigh: for symmetric matrices, sorted ascending
order = np.argsort(eigenvalues)[::-1]
eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]

explained = eigenvalues / eigenvalues.sum()
print('variance explained by the first five directions:', np.round(explained[:5], 3))
print(f'first 10 directions explain {explained[:10].sum():.1%}; first 20 explain {explained[:20].sum():.1%}; first 40 explain {explained[:40].sum():.1%}')
print(f'{np.sum(eigenvalues < 1e-10)} directions have no variance at all (pixels that are always zero)')

In [ ]:
components = eigenvectors[:, :2]
Z = Xc @ components                                          # project every digit onto the two main directions
print('the two directions are perpendicular:', np.round(components[:, 0] @ components[:, 1], 10))
print('variance of the projected data matches the eigenvalues:', np.round(Z.var(axis=0, ddof=1), 2), np.round(eigenvalues[:2], 2))

## 3. The one line you would ship

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2).fit(X)
Z_sk = pca.transform(X)
print('scikit-learn explained variance:', np.round(pca.explained_variance_ratio_, 3), ' ours:', np.round(explained[:2], 3))
print('same projection, up to sign:', np.allclose(np.abs(Z_sk), np.abs(Z), atol=1e-6))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(Z_sk[:, 0], Z_sk[:, 1], c=y, cmap='tab10', s=10)
plt.colorbar(sc, ticks=range(10), label='digit')
ax.set_xlabel('first component'); ax.set_ylabel('second component'); ax.set_title('1,797 digits, 64 pixels, two numbers each')
plt.show()

## 4. Reconstruction: how much do you lose?

In [ ]:
def reconstruct(X, k):
    p = PCA(n_components=k).fit(X)
    return p.inverse_transform(p.transform(X))

print(f'{"components":>10} {"pixel error":>12} {"variance kept":>14}')
for k in [1, 5, 10, 20, 40, 64]:
    Xr = reconstruct(X, k)
    err = np.mean(np.abs(Xr - X))
    print(f'{k:>10} {err:>12.2f} {explained[:k].sum():>14.1%}')

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for col, k in enumerate([1, 5, 10, 20, 40, 64]):
    Xr = reconstruct(X, k)
    for row, idx in enumerate([0, 7]):
        axes[row, col].imshow(Xr[idx].reshape(8, 8), cmap='gray_r', vmin=0, vmax=16)
        axes[row, col].set_title(f'{k} comp.' if row == 0 else ''); axes[row, col].axis('off')
plt.suptitle('the same two digits rebuilt from k components'); plt.show()

## 5. PCA as a preprocessing step

In [ ]:
import time
from sklearn.pipeline import make_pipeline

print(f'{"features":>9} {"k-NN accuracy":>14} {"time":>8}')
for k in [None, 40, 20, 10, 5]:
    model = KNeighborsClassifier(5) if k is None else make_pipeline(PCA(n_components=k), KNeighborsClassifier(5))
    start = time.perf_counter()
    acc = cross_val_score(model, X, y, cv=5).mean()
    print(f'{str(k if k else 64):>9} {acc:>14.3f} {time.perf_counter() - start:>7.2f}s')

In [ ]:
# And on the digits with 1,024 noise columns stapled on
Xn = np.column_stack([X, rng.normal(0, 4, (len(X), 1024))])
for name, model in [('k-NN on all 1,088 columns', KNeighborsClassifier(5)),
                    ('PCA to 20, then k-NN', make_pipeline(PCA(n_components=20), KNeighborsClassifier(5)))]:
    print(f'{name:<28} {cross_val_score(model, Xn, y, cv=5).mean():.3f}')

## 6. Scale first, or the loudest feature wins

In [ ]:
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(3)
n = 500
hours = rng.uniform(0, 10, n)
spend = 2000 * hours + rng.normal(0, 3000, n)                 # rupees: large numbers, loosely tied to hours
visits = rng.integers(1, 20, n).astype(float)                 # unrelated to either
F = np.column_stack([hours, spend, visits])
for name, data in [('raw', F), ('standardised', StandardScaler().fit_transform(F))]:
    p = PCA().fit(data)
    print(f'{name:<13} variance explained {np.round(p.explained_variance_ratio_, 3)}   first direction {np.round(p.components_[0], 3)}')

## 7. Variance is not the same as usefulness

In [ ]:
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(4)
n = 1000
loud = rng.normal(0, 10, (n, 3))                              # three big, noisy, useless features
quiet = rng.normal(0, 1, n)                                   # one small feature that decides the label
label = (quiet > 0).astype(int)
G = np.column_stack([loud, quiet])
p = PCA(n_components=1).fit(G)
print('first component weights (three loud features, then the quiet one):', np.round(p.components_[0], 3))
one = cross_val_score(LogisticRegression(), p.transform(G), label, cv=5).mean()
every = cross_val_score(LogisticRegression(), G, label, cv=5).mean()
print(f'logistic regression on the first component: {one:.3f}   on all four features: {every:.3f}')

## 8. Supervised reduction: LDA knows the label

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda = LinearDiscriminantAnalysis(n_components=2).fit(X, y)
for name, Z2 in [('PCA, 2 components', Z_sk), ('LDA, 2 components', lda.transform(X))]:
    acc = cross_val_score(KNeighborsClassifier(5), Z2, y, cv=5).mean()
    print(f'{name:<20} k-NN accuracy in two dimensions {acc:.3f}')

## 9. t-SNE: pictures, and the traps in them

In [ ]:
from sklearn.manifold import TSNE

sub = rng.choice(len(X), 600, replace=False)                  # t-SNE is slow; use a sample
start = time.perf_counter()
T = TSNE(n_components=2, perplexity=30, init='random', random_state=0).fit_transform(X[sub])
print(f't-SNE on 600 digits took {time.perf_counter() - start:.1f} s')
print(f'k-NN accuracy in the t-SNE plane: {cross_val_score(KNeighborsClassifier(5), T, y[sub], cv=5).mean():.3f}   in the PCA plane: {cross_val_score(KNeighborsClassifier(5), Z_sk[sub], y[sub], cv=5).mean():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
axes[0].scatter(Z_sk[sub, 0], Z_sk[sub, 1], c=y[sub], cmap='tab10', s=10); axes[0].set_title('PCA')
axes[1].scatter(T[:, 0], T[:, 1], c=y[sub], cmap='tab10', s=10); axes[1].set_title('t-SNE')
plt.show()

In [ ]:
# Trap one: run it twice, get two different pictures
T2 = TSNE(n_components=2, perplexity=30, init='random', random_state=1).fit_transform(X[sub])
print(f'correlation between the two runs\' first axes: {np.corrcoef(T[:, 0], T2[:, 0])[0, 1]:.2f}')

# Trap two: distances between clusters mean nothing. Three blobs, one far away.
from sklearn.datasets import make_blobs
B, b = make_blobs(n_samples=300, centers=[[0, 0], [3, 0], [30, 0]], cluster_std=0.5, random_state=0)
TB = TSNE(n_components=2, perplexity=30, random_state=0).fit_transform(B)
def centre_gap(Z, i, j): return np.linalg.norm(Z[b == i].mean(axis=0) - Z[b == j].mean(axis=0))
print(f'original: gap between blobs 0 and 1 is {centre_gap(B, 0, 1):.1f}, between 0 and 2 is {centre_gap(B, 0, 2):.1f}  (ten times further)')
print(f't-SNE   : gap between blobs 0 and 1 is {centre_gap(TB, 0, 1):.1f}, between 0 and 2 is {centre_gap(TB, 0, 2):.1f}')

# Trap three: perplexity changes the picture
for perp in [5, 30, 100]:
    Tp = TSNE(n_components=2, perplexity=perp, random_state=0).fit_transform(X[sub])
    print(f'perplexity {perp:>3}: k-NN accuracy in the plane {cross_val_score(KNeighborsClassifier(5), Tp, y[sub], cv=5).mean():.3f}')

## Try it yourself

1. In section 4, find the smallest number of components at which you can still read both digits in the picture. How much variance is that?
2. In section 6, multiply `hours` by 1,000 so it is in seconds. Which direction does raw PCA now find?
3. In section 9, run t-SNE on 600 rows of pure noise, `rng.normal(size=(600, 64))`. Does the picture have clusters in it?